In [116]:
import numpy as np
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

# FASHIONMNIST 0.6

In [83]:
MNIST_03_FILE_PATH = 'data/FashionMNIST0.3.npz'
MNIST_06_FILE_PATH = 'data/FashionMNIST0.6.npz'
CIFAR_FILE_PATH = 'data/CIFAR.npz'

## load data, prepare images, check prepared images

Xtr: training images.

Str: noisy training labels.

Xts: test images.

Yts: clean test labels.

In [84]:
dataset = np.load(MNIST_03_FILE_PATH)
Xtr = dataset['Xtr']
Str = dataset['Str']
Xts = dataset['Xts']
Yts = dataset['Yts']
print(Xtr.shape)
print(Str.shape)
print(Xts.shape)
print(Yts.shape)

(18000, 784)
(18000,)
(3000, 784)
(3000,)


In [85]:
dataset

NpzFile 'data/FashionMNIST0.3.npz' with keys: Xtr, Str, Xts, Yts

In [86]:
print("Noisy labels: ",set(Str))

Noisy labels:  {np.uint8(0), np.uint8(1), np.uint8(2)}


In [87]:
def prepare_images(images):

    # Convert to float32
    images = images.astype(np.float32)

    if images.ndim == 2 and images.shape[1] == 784:
        # Fashion: [N, 784] -> [N, 1, 28, 28]
        images = images.reshape(images.shape[0], 1, 28, 28)

    elif images.ndim == 4 and images.shape[-1] == 3:
        # CIFAR: [N, 32, 32, 3] -> [N, 3, 32, 32]
        images = images.reshape(images.shape[0], 3, 32, 32)

    else:
        raise ValueError(f"Unexpected image shape: {images.shape}")

    # Change pixel range from [0, 255] to [0, 1]
    images = images / 255

    # Convert NumPy array to a PyTorch tensor
    images = torch.from_numpy(images)

    return images

In [88]:
def load_data(path):
    dataset = np.load(path)
    return dataset

In [89]:
fashion_03 = load_data(MNIST_03_FILE_PATH)
fashion_06 = load_data(MNIST_06_FILE_PATH)
cifar = load_data(CIFAR_FILE_PATH)

In [91]:
fashion_images = prepare_images(fashion_06["Xtr"])

print("Shape:", fashion_images.shape)
print("Type:", fashion_images.dtype)
print("Minimum:", fashion_images.min())
print("Maximum:", fashion_images.max())

Shape: torch.Size([18000, 1, 28, 28])
Type: torch.float32
Minimum: tensor(0.)
Maximum: tensor(1.)


In [92]:
assert fashion_images.ndim == 4
assert fashion_images.shape[1:] == (1, 28, 28)
assert fashion_images.dtype == torch.float32
assert torch.isfinite(fashion_images).all()
assert fashion_images.min() >= 0
assert fashion_images.max() <= 1

## Prepare labels

In [93]:
def prepare_labels(labels):
    # Convert labels to a PyTorch long-integer tensor
    labels = torch.tensor(labels, dtype=torch.long)

    # Check that it is a one-dimensional array
    assert labels.ndim == 1

    # Check that the labels are only 0, 1, or 2
    assert np.isin(labels, [0, 1, 2]).all()

    return labels

In [94]:
print("Str shape:", fashion_03["Str"].shape)
print("Str dtype:", fashion_03["Str"].dtype)
print("Str minimum:", fashion_03["Str"].min())
print("Str maximum:", fashion_03["Str"].max())
print("Str unique values:", np.unique(fashion_03["Str"]))

print("Yts unique values:", np.unique(fashion_03["Yts"]))

Str shape: (18000,)
Str dtype: uint8
Str minimum: 0
Str maximum: 2
Str unique values: [0 1 2]
Yts unique values: [0 1 2]


In [95]:
Xtr = prepare_images(fashion_03["Xtr"])
Str = prepare_labels(fashion_03["Str"])

Xts = prepare_images(fashion_03["Xts"])
Yts = prepare_labels(fashion_03["Yts"])

## Split data to 80 train & 20 val

In [96]:
def make_split(num_examples, seed, validation_fraction=0.20):
    # Create a NumPy random-number generator using the seed
    rng = np.random.default_rng(seed)

    # Randomly reorder all indices from 0 to num_examples - 1
    indices = rng.permutation(num_examples)

    # Calculate how many validation examples are needed
    validation_size = int(num_examples * validation_fraction)

    # Take one part for validation and the rest for training
    validation_indices = indices[:validation_size]
    training_indices = indices[validation_size:]

    assert np.intersect1d(
        training_indices, validation_indices
    ).size == 0

    assert np.array_equal(
        np.sort(np.concatenate((training_indices, validation_indices))),
        np.arange(num_examples),
    )

    return training_indices, validation_indices

In [97]:
train_indices, val_indices = make_split(
    num_examples=len(fashion_images),
    seed=7,
)

assert len(train_indices) == 14400
assert len(val_indices) == 3600
assert len(set(train_indices) & set(val_indices)) == 0
assert len(train_indices) + len(val_indices) == 18000

## Create pytorch dataset

In [98]:
class IndexedNoisyDataset(Dataset):
    def __init__(self, images, noisy_labels, indices):
        self.images = images
        self.noisy_labels = noisy_labels
        self.indices = torch.as_tensor(indices, dtype=torch.long)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        original_index = self.indices[position]

        image = self.images[original_index]
        label = self.noisy_labels[original_index]

        return image, label

## Create Loaders

In [99]:
train_dataset = IndexedNoisyDataset(
    images=Xtr,
    noisy_labels=Str,
    indices=train_indices,
)

validation_dataset = IndexedNoisyDataset(
    images=Xtr,
    noisy_labels=Str,
    indices=val_indices,
)

In [100]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label:", label)
print("Label shape:", label.shape)
print("Image dtype:", image.dtype)
print("Label dtype:", label.dtype)

Image shape: torch.Size([1, 28, 28])
Label: tensor(1)
Label shape: torch.Size([])
Image dtype: torch.float32
Label dtype: torch.int64


In [101]:
original_index = train_indices[0]

assert torch.equal(
    train_dataset[0][0],
    Xtr[original_index],
)

assert train_dataset[0][1] == Str[original_index]

In [102]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)

In [103]:
batch_images, batch_labels = next(iter(train_loader))

print("Image batch:", batch_images.shape)
print("Label batch:", batch_labels.shape)
print("Image type:", batch_images.dtype)
print("Label type:", batch_labels.dtype)
print("Pixel range:", batch_images.min(), batch_images.max())
print("Labels:", torch.unique(batch_labels))

Image batch: torch.Size([64, 1, 28, 28])
Label batch: torch.Size([64])
Image type: torch.float32
Label type: torch.int64
Pixel range: tensor(0.) tensor(1.)
Labels: tensor([0, 1, 2])


In [104]:
assert batch_images.shape == (64, 1, 28, 28)
assert batch_labels.shape == (64,)
assert batch_images.dtype == torch.float32
assert batch_labels.dtype == torch.int64
assert torch.isfinite(batch_images).all()

## Classif 1: CNN

In [73]:
class SmallCNN(nn.Module):
    def __init__(self, input_channels, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=input_channels,
                out_channels=16,     # Hint: 16
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=16,      # Must match the previous out_channels
                out_channels=32,     # Hint: 32
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(
            in_features=32,        # Number of final feature channels
            out_features=num_classes,
        )

    def forward(self, x):
        # [B, C, H, W] -> feature maps
        x = self.features(x)

        # Feature maps -> [B, final_channels, 1, 1]
        x = self.pool(x)

        # [B, final_channels, 1, 1] -> [B, final_channels]
        x = torch.flatten(x, start_dim=1)

        # Produce three logits
        logits = self.classifier(x)

        return logits

In [74]:
batch_images, batch_labels = next(iter(train_loader))

print("Input shape:", batch_images.shape)

Input shape: torch.Size([64, 1, 28, 28])


In [75]:
model = SmallCNN(
    input_channels=1,
    num_classes=3,
)

In [76]:
logits = model(batch_images)

print("Output shape:", logits.shape)
print("First output:", logits[0])

Output shape: torch.Size([64, 3])
First output: tensor([ 0.1534,  0.0854, -0.0585], grad_fn=<SelectBackward0>)


In [117]:
probabilities = torch.softmax(logits, dim=1)

print("First probabilities:", probabilities[0])
print("Sum:", probabilities[0].sum())

First probabilities: tensor([0.3599, 0.3459, 0.2942], grad_fn=<SelectBackward0>)
Sum: tensor(1., grad_fn=<SumBackward0>)


In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
transforms = weights.transforms()


## Transition matrix

Fashion MNIST 0.3 has defined transition matrix so we'll do it first

In [78]:
T = torch.tensor([
    [0.7, 0.3, 0.0],
    [0.0, 0.7, 0.3],
    [0.3, 0.0, 0.7],
], dtype=torch.float32)

In [79]:
print("Shape:", T.shape)
print("Row sums:", T.sum(dim=1))
print("Minimum:", T.min())
print("Maximum:", T.max())

Shape: torch.Size([3, 3])
Row sums: tensor([1., 1., 1.])
Minimum: tensor(0.)
Maximum: tensor(0.7000)


In [80]:
assert T.shape == (3, 3)
assert torch.isfinite(T).all()
assert torch.all(T >= 0)
assert torch.all(T <= 1)

assert torch.allclose(
    T.sum(dim=1),
    torch.ones(3),
    atol=1e-6,
)

In [81]:
probabilities = torch.softmax(logits, dim=1)

In [82]:
p = probabilities
q = p @ T

In [50]:
print("p shape:", p.shape)
print("q shape:", q.shape)
print("First p:", p[0])
print("First q:", q[0])
print("First q sum:", q[0].sum())

assert q.shape == (64, 3)
assert torch.isfinite(q).all()
assert torch.all(q >= 0)

assert torch.allclose(
    q.sum(dim=1),
    torch.ones(64),
    atol=1e-6,
)

p shape: torch.Size([64, 3])
q shape: torch.Size([64, 3])
First p: tensor([0.3859, 0.2800, 0.3340], grad_fn=<SelectBackward0>)
First q: tensor([0.3704, 0.3118, 0.3178], grad_fn=<SelectBackward0>)
First q sum: tensor(1., grad_fn=<SumBackward0>)


## Forward correction

In [ ]:
def forward_correction_loss(
    logits,
    noisy_labels,
    transition_matrix,
    epsilon=1e-8,
):
    # Step 1: logits -> clean probabilities p
    p = torch.nn.functional.softmax(logits, dim=1)

    # Step 2: apply the transition matrix
    q = p @ transition_matrix

    # Step 3: change labels from [B] to [B, 1]
    label_column = noisy_labels.unsqueeze(1)

    # Step 4: select q[b, noisy_labels[b]]
    selected_probabilities = q.gather(
        dim=1,
        index=label_column,
    ).squeeze(1)

    # Step 5: protect against log(0)
    selected_probabilities = selected_probabilities.clamp_min(epsilon)

    # Step 6: calculate one loss per image
    individual_losses = -torch.log(selected_probabilities)

    # Step 7: average the losses
    loss = individual_losses.mean()

    return loss

In [110]:
loss = forward_correction_loss(
    logits=logits,
    noisy_labels=batch_labels,
    transition_matrix=T,
)

print("Loss:", loss)
print("Loss shape:", loss.shape)

assert loss.ndim == 0
assert torch.isfinite(loss)
assert loss.item() >= 0

Loss: tensor(1.0990, grad_fn=<MeanBackward0>)
Loss shape: torch.Size([])


In [112]:
T = T.to(logits.device)

loss = forward_correction_loss(
    logits,
    batch_labels,
    T,
)

print("Loss:", loss)
print("Loss shape:", loss.shape)

assert loss.ndim == 0
assert torch.isfinite(loss)
assert loss.item() >= 0

Loss: tensor(1.0990, grad_fn=<MeanBackward0>)
Loss shape: torch.Size([])


In [113]:
identity = torch.eye(
    3,
    device=logits.device,
    dtype=logits.dtype,
)

forward_loss = forward_correction_loss(
    logits,
    batch_labels,
    identity,
)

ordinary_loss = nn.functional.cross_entropy(
    logits,
    batch_labels,
)

print("Forward loss:", forward_loss.item())
print("Ordinary loss:", ordinary_loss.item())
print("Difference:", abs(forward_loss.item() - ordinary_loss.item()))

assert torch.allclose(
    forward_loss,
    ordinary_loss,
    atol=1e-6,
)

Forward loss: 1.1007843017578125
Ordinary loss: 1.100784420967102
Difference: 1.1920928955078125e-07


In [114]:
model.zero_grad()

logits = model(batch_images)
loss = forward_correction_loss(logits, batch_labels, T)
loss.backward()

for name, parameter in model.named_parameters():
    if parameter.grad is not None:
        assert torch.isfinite(parameter.grad).all(), name

print("Finite gradients: passed")

Finite gradients: passed


In [115]:
assert T.requires_grad is False